# Milestone 2: Facial Recognition Model Evaluation Report 
**System:** Intelligent Shop Security

## Objective
The primary objective of this notebook is to evaluate the developed facial recognition model's ability to efficiently detect and recognize faces.  
This document serves as the official **Model Evaluation Report** detailing the model's performance metrics.

## Model Selection
For this project, we selected **FaceNet**, a widely-used model for face recognition that produces compact facial embeddings.  
To streamline deployment and evaluation, we are utilizing the **DeepFace** framework, which efficiently wraps several state-of-the-art face recognition models including FaceNet. 

This approach leverages transfer learning via a pre-trained model , optimized for real-time performance while maintaining high recognition accuracy.

In [ ]:
# 1. Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.spatial.distance import cosine
from deepface import DeepFace
import os
import cv2

print("Loading FaceNet Model via DeepFace framework...")
# Load the model directly
embedding_model = DeepFace.build_model("Facenet").model
print("Model loaded successfully and ready for evaluation!")

C:\Users\eslam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



⏳ Loading FaceNet Model via DeepFace framework...
✅ Model loaded successfully and ready for evaluation!


## Evaluation Methodology

To accurately assess the model, we evaluate both **identification** (who the person is) and **verification** (is this the same person as before?). 

The evaluation relies on generating positive pairs (images of the same person) and negative pairs (images of different people) from our preprocessed dataset. We then calculate the Cosine Distance between their embeddings.

We will calculate the following key performance metrics:
* **Accuracy:** Overall correctness of the model.
* **Precision and Recall**.
* **F1-score**.
* **False Acceptance Rate (FAR):** A critical metric for security-focused systems, representing the probability that the system incorrectly authorizes an unauthorized person.

### Mathematical Definitions:
* $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$

* $Precision = \frac{TP}{TP + FP}$

* $Recall = \frac{TP}{TP + FN}$

* $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$

* $FAR = \frac{FP}{FP + TN}$

* $FRR = \frac{FN}{FN + TP}$

In [ ]:
# 2. Helper functions for evaluation

def get_embedding(img_path):
    """Extracts the 128D embedding for a given image path."""
    try:
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (160, 160))
        img_array = np.expand_dims(img_resized, axis=0) / 255.0
        return embedding_model.predict(img_array, verbose=0)[0]
    except Exception as e:
        return None

def evaluate_pairs(pairs, labels, threshold=0.40):
    """
    Evaluates a list of image pairs.
    labels: 1 if same person (Genuine), 0 if different people (Impostor).
    """
    predictions = []
    distances = []
    
    print(f"Evaluating {len(pairs)} pairs...")
    for (img1_path, img2_path) in pairs:
        emb1 = get_embedding(img1_path)
        emb2 = get_embedding(img2_path)
        
        if emb1 is not None and emb2 is not None:
            dist = cosine(emb1, emb2)
            distances.append(dist)
            predictions.append(1 if dist < threshold else 0)
        else:
            predictions.append(0)
            distances.append(1.0)
            
    return np.array(predictions), np.array(distances)

## 📈 Calculating Performance Metrics
In this section, we run the evaluation on our test dataset. 

*(Note: Replace the dummy `test_pairs` below with the actual pairs generated from the `data/processed_dataset/` provided by the Data Collection team).*

In [ ]:
import os
import random
import itertools

def generate_image_pairs(dataset_path, num_positive=150, num_negative=150):
    print(f"Reading dataset from: {dataset_path}")
    
    if not os.path.exists(dataset_path):
        print("Path not found! Make sure M1 team delivered the data.")
        return [], []
        
    classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    
    class_images = {}
    for cls in classes:
        img_dir = os.path.join(dataset_path, cls)
        images = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if len(images) > 0:
            class_images[cls] = images
            
    valid_classes = [c for c in class_images if len(class_images[c]) >= 2]
    
    pairs = []
    labels = []
    
    if valid_classes:
        for _ in range(num_positive):
            cls = random.choice(valid_classes)
            img1, img2 = random.sample(class_images[cls], 2)
            pairs.append((img1, img2))
            labels.append(1)
            
    all_classes = list(class_images.keys())
    if len(all_classes) >= 2:
        for _ in range(num_negative):
            cls1, cls2 = random.sample(all_classes, 2)
            img1 = random.choice(class_images[cls1])
            img2 = random.choice(class_images[cls2])
            pairs.append((img1, img2))
            labels.append(0)
            
    print(f"✅ Successfully generated {len(pairs)} pairs ({labels.count(1)} Positive, {labels.count(0)} Negative).")
    return pairs, np.array(labels)


DATASET_PATH = "../data/processed_dataset/" 

test_pairs, y_true = generate_image_pairs(DATASET_PATH, num_positive=100, num_negative=100)

if len(test_pairs) > 0:
    y_pred, distances = evaluate_pairs(test_pairs, y_true, threshold=0.40)

📂 Reading dataset from: ../data/processed_dataset/
✅ Successfully generated 0 pairs (0 Positive, 0 Negative).


In [5]:
# Calculate Standard Metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

# Calculate Confusion Matrix to extract TP, TN, FP, FN
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Calculate Security-Specific Metrics
far = fp / (fp + tn) # False Acceptance Rate
frr = fn / (fn + tp) # False Rejection Rate

print("📊 --- Model Evaluation Results --- 📊")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1:.4f}")
print("-" * 30)
print(f"🚨 False Acceptance Rate (FAR): {far * 100:.2f}%")
print(f"🔒 False Rejection Rate (FRR): {frr * 100:.2f}%")

NameError: name 'y_pred' is not defined